# AI Assistant: Card Play Analysis

This notebook uses the AI Assistant to analyze card play decisions during a trick.

For each card you can play, the AI shows a win probability percentage.


In [1]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

from eucher.ai_players.ai_assistant import AIAssistant
from eucher.cards import Card, Deck, Rank, Suit
from eucher.rules import RulesEngine
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets


# Monkey patch PyTorchStrategicPlayer to add missing abstract methods
from eucher.ai_players.pytorch_player import PyTorchStrategicPlayer

if not hasattr(PyTorchStrategicPlayer, 'decide_going_alone'):
    def decide_going_alone(self, player, trump_suit):
        """Decide whether to go alone. Default: False."""
        return False
    PyTorchStrategicPlayer.decide_going_alone = decide_going_alone

if not hasattr(PyTorchStrategicPlayer, 'decide_trade_in'):
    def decide_trade_in(self, player, eligible_cards):
        """Decide whether to trade in. Default: False."""
        return False
    PyTorchStrategicPlayer.decide_trade_in = decide_trade_in
# Monkey patch PyTorchStrategicPlayer to add missing abstract methodsfrom eucher.ai_players.pytorch_player import PyTorchStrategicPlayerif not hasattr(PyTorchStrategicPlayer, 'decide_going_alone'):    def decide_going_alone(self, player, trump_suit):        """Decide whether to go alone. Default: False."""        return False    PyTorchStrategicPlayer.decide_going_alone = decide_going_aloneif not hasattr(PyTorchStrategicPlayer, 'decide_trade_in'):    def decide_trade_in(self, player, eligible_cards):        """Decide whether to trade in. Default: False."""        return False    PyTorchStrategicPlayer.decide_trade_in = decide_trade_in

In [2]:
# Initialize AI Assistant
thinking_time_dropdown = widgets.Dropdown(
    options=['fast', 'quick', 'normal', 'thorough', 'deep'],
    value='normal',
    description='Thinking Time:',
    style={'description_width': 'initial'}
)

assistant = AIAssistant(thinking_time=thinking_time_dropdown.value)
rules = RulesEngine()

def update_thinking_time(change):
    global assistant
    assistant = AIAssistant(thinking_time=change['new'])

thinking_time_dropdown.observe(update_thinking_time, names='value')

display(thinking_time_dropdown)


Dropdown(description='Thinking Time:', index=2, options=('fast', 'quick', 'normal', 'thorough', 'deep'), style…

In [3]:
# Example: Analyze card play decision
deck = Deck()
deck.shuffle()
hand = deck.deal(5)
trump_suit = Suit.HEARTS
led_suit = Suit.SPADES
trick_cards = [Card(Suit.SPADES, Rank.ACE)]  # Opponent led ace of spades

valid_cards = rules.get_valid_plays(hand, led_suit, trump_suit)

display(HTML(f"""
<div style="border: 2px solid #333; padding: 10px; margin: 10px;">
    <h3>Card Play Decision Analysis</h3>
    <p><strong>Your Hand:</strong> {', '.join(str(c) for c in hand)}</p>
    <p><strong>Trump Suit:</strong> {trump_suit.name}</p>
    <p><strong>Led Suit:</strong> {led_suit.name}</p>
    <p><strong>Trick Cards:</strong> {', '.join(str(c) for c in trick_cards)}</p>
    <p><strong>Valid Cards:</strong> {', '.join(str(c) for c in valid_cards)}</p>
</div>
"""))

analyze_button = widgets.Button(description="Analyze Card Play Options", button_style='success')
result_output = widgets.Output()

def analyze_card_play(b):
    with result_output:
        clear_output()
        display(HTML("<p>Running Monte Carlo simulation for each card... This may take a moment.</p>"))
        
        results = assistant.analyze_card_play(
            hand=hand,
            valid_cards=valid_cards,
            led_suit=led_suit,
            trump_suit=trump_suit,
            trick_cards=trick_cards,
            player_id=0,
            team=0,
            trick_number=1,
            tricks_won_team0=0,
            tricks_won_team1=0
        )
        
        # Sort by win probability
        results.sort(key=lambda x: x[1], reverse=True)
        
        # Build HTML table
        rows = []
        for card, win_prob in results:
            color = 'green' if win_prob >= 0.7 else 'orange' if win_prob >= 0.5 else 'red'
            rows.append(f"""
            <tr>
                <td>{card}</td>
                <td style="color: {color}; font-weight: bold;">{win_prob*100:.1f}%</td>
            </tr>
            """)
        
        clear_output()
        display(HTML(f"""
        <div style="border: 2px solid #333; padding: 10px; margin: 10px;">
            <h3>Card Play Analysis Results</h3>
            <table style="width: 100%; border-collapse: collapse;">
                <thead>
                    <tr style="background-color: #f0f0f0;">
                        <th style="padding: 8px; border: 1px solid #ddd;">Card</th>
                        <th style="padding: 8px; border: 1px solid #ddd;">Win %</th>
                    </tr>
                </thead>
                <tbody>
                    {''.join(rows)}
                </tbody>
            </table>
            <p style="margin-top: 10px;"><strong>Recommended:</strong> {results[0][0]} ({results[0][1]*100:.1f}% win probability)</p>
        </div>
        """))

analyze_button.on_click(analyze_card_play)
display(analyze_button)
display(result_output)


Button(button_style='success', description='Analyze Card Play Options', style=ButtonStyle())

Output()